In [1]:
from tool_server.utils.utils import *
from tool_server.utils.prompts import tool_planning_model_prompt_one_tool_call
from tqdm import tqdm
import json, os
from copy import deepcopy
metric_dict = {
    "chartqa": ["Exact_Match_Acc"],
    "chartgemma": ["Acc"],
    "charxiv": ["type_res/descriptive", "type_res/reasoning"],
    "docvqa": ["exact_score"],
    "ocrbench": ["Acc"],
    "reachqa": ["chart_type_res/AVG"],
    "vstar": ["category_res/direct_attributes","category_res/relative_position"],
}

model_name_dict = {
    "qwen25vl3b": "Qwen-2.5VL-3B /w Tools",
    "qwen25vl3b_sftv1": "Qwen-2.5VL-3B SFT v1",
    "qwen25vl3b_notool": "Qwen-2.5VL-3B /wo Tools",
    "qwen25vl7b": "Qwen-2.5VL-7B /w Tools",
    "qwen25vl7b_sftv1": "Qwen-2.5VL-7B SFT v1",
    "qwen25vl7b_notool": "Qwen-2.5VL-7B /wo Tools",
}
task_list = ["chartqa", "chartgemma", "charxiv", "docvqa", "ocrbench", "reachqa","vstar"]
res_base_dir = "/mnt/petrelfs/songmingyang/code/reasoning/tool-agent/tool_server/tf_eval/scripts/logs/results"


In [2]:

def get_res_dict(model_name_dict, res_base_dir = "/mnt/petrelfs/songmingyang/code/reasoning/tool-agent/tool_server/tf_eval/scripts/logs/results"):
    res_dict = {}
    for model_name,full_name in model_name_dict.items():
        result_path = os.path.join(res_base_dir, model_name, "qwen25_allres.jsonl")
        result_data = process_jsonl(result_path)
        result_dict = {}
        for result_item in result_data:
            task_name = result_item["task_name"]
            result_dict[task_name] = result_item
        res_dict[full_name] = result_dict
    return res_dict
        
def print_res_table(big_res_dict, task_list, metric_dict=metric_dict):
    res = ""
    for model_name, res_dict in big_res_dict.items():
        res += f"\\textbf{{{model_name}}}"
        
        for task_name in task_list:
            if task_name in res_dict:
                result_item = res_dict[task_name]
                if "task_name" not in result_item:
                    print(f"Warning: task_name not found in result_item for {model_name} and {task_name}")
                acc = get_result_number(result_item)
                for acc_idx,acc in enumerate(acc):
                    # all_res = [get_result_number(v.get(task_name,{}))[acc_idx] for k,v in res_dict.items()]
                    # all_res.sort()
                    all_res = [-1,-2]
                    if acc == all_res[-1]:
                        res += f" & \\textbf{acc*100:.2f}" 
                    elif acc == all_res[-2]:
                        res += f" & \\underline{acc*100:.2f}" 
                    else:
                        res += f" & {acc*100:.2f}"
            else:
                metric_num = len(metric_dict[task_name])
                for _ in range(metric_num):
                    res += " & -"
        res += " \\\\ \n"
    return res

def get_result_number(result_item, metric_dict=metric_dict):
    res = []
    # print(result_item)   
    task_name = result_item["task_name"]
    display_metrics = metric_dict.get(task_name, [])
    for display_metric in display_metrics:
        waydown = display_metric.split("/")
        current_result_item = deepcopy(result_item)
        for way in waydown:
            
            if way == "AVG":
                current_res_dict_list = [v for k,v in current_result_item.items()]
                average = sum(current_res_dict_list) / len(current_res_dict_list) if current_res_dict_list else 0.0
                res.append(average)
                break
            
            current_result_item = current_result_item[way]
            if isinstance(current_result_item, dict):
                continue
            else:
                res.append(current_result_item)
                break
    return res
            

In [3]:
res_dict = get_res_dict(model_name_dict, res_base_dir)


In [4]:
res = print_res_table(res_dict, task_list)
print(res)

\textbf{Qwen-2.5VL-3B /w Tools} & 0.00 & 7.34 & 30.20 & 8.30 & 3.68 & 69.00 & 8.01 & 29.57 & 50.00 \\ 
\textbf{Qwen-2.5VL-3B SFT v1} & 8.72 & 5.23 & 9.43 & 6.50 & 8.24 & 45.80 & 3.59 & 51.30 & 53.95 \\ 
\textbf{Qwen-2.5VL-3B /wo Tools} & 0.08 & 11.37 & 29.63 & 14.20 & 17.52 & 70.10 & 8.66 & 43.48 & 57.89 \\ 
\textbf{Qwen-2.5VL-7B /w Tools} & 0.00 & 11.87 & 57.27 & 13.70 & 7.20 & 68.90 & 13.57 & 50.43 & 64.47 \\ 
\textbf{Qwen-2.5VL-7B SFT v1} & 7.64 & 9.15 & 25.87 & 10.50 & 13.78 & 52.90 & 6.00 & 65.22 & 65.79 \\ 
\textbf{Qwen-2.5VL-7B /wo Tools} & 0.00 & 32.80 & 66.33 & 27.80 & 30.66 & 79.00 & 12.68 & 64.35 & 68.42 \\ 



In [5]:
metric_dict

{'chartqa': ['Exact_Match_Acc'],
 'chartgemma': ['Acc'],
 'charxiv': ['type_res/descriptive', 'type_res/reasoning'],
 'docvqa': ['exact_score'],
 'ocrbench': ['Acc'],
 'reachqa': ['chart_type_res/AVG'],
 'vstar': ['category_res/direct_attributes', 'category_res/relative_position']}